# Робот, камера и карта

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elena-kornaeva/AGLA/blob/main/weeks/week-06-coordinate-systems/notebooks/robot-camera-map.ipynb)

Практика к лекции № 4 · редакционная неделя 06 · 35–45 минут + 15–25 минут исследования.

**Вопрос:** почему правильные числа датчика могут привести робота не туда?
После работы вы сможете переводить координаты точки и перемещения, отличать движение от смены описания и измерять длину в косоугольном базисе.

Это синтетическая 2D-комната с уже найденными маяками, а не перспективная камера. Внешних данных и изображений нет. Источник замысла: [сценарий занятия](../../../docs/notebook-scenarios.md#1-робот-камера-и-карта); [соглашения](../../../docs/notation.md). Исходная презентация недоступна: её дополнительные детали не воспроизводятся.

Запускайте ячейки по порядку. В упражнениях заменяйте `None` своим решением и повторно запускайте проверку. Пустые упражнения не прерывают запуск. Python 3, NumPy, Matplotlib; установки, сеть и GPU не нужны.

## 1. Робот приехал не туда · 7 минут
Камера прислала пакет `(3, 1)`. Навигация прочитала его как точку карты. Маяк A на карте расположен в `(1, 3)`.

**До запуска графика:** отметьте на бумаге обе точки. Если повернуть камеру, не двигая маяки, какие числа изменятся? Оцените промах.

**Мой прогноз:** …

Следующая ячейка содержит служебное рисование. Её можно запустить, не разбирая реализацию.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")

def checked_basis(P):
    P = np.asarray(P, dtype=float)
    if P.shape != (2, 2) or not np.isfinite(P).all():
        raise ValueError("Нужна конечная матрица 2×2.")
    condition = np.linalg.cond(P)
    if not np.isfinite(condition) or condition >= 1e10:
        raise ValueError("Базис вырожден или почти вырожден: надёжный перевод остановлен.")
    return P

def rotation(degrees):
    t = np.deg2rad(degrees)
    return np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])

def scene(a=(2, 0), angle=90, target="A", corrected=False, skew=0., decompose=False):
    # Служебный перевод для визуализации; функции студента появятся ниже.
    a = np.asarray(a, dtype=float)
    P = checked_basis(rotation(angle) @ np.array([[1., skew], [0., 1.]]))
    beacons = {"A": np.array([1., 3.]), "B": np.array([4., 3.])}
    x = beacons[target]
    y = np.linalg.solve(P, x-a)
    wrong = y.copy()
    destination = x if corrected else wrong
    robot = np.array([0., 0.])
    room = np.array([[-1,-1], [6,-1], [6,5], [-1,5], [-1,-1]], float)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), layout="constrained")
    transforms = [lambda v: np.asarray(v), lambda v: np.linalg.solve(P, np.asarray(v)-a)]
    for k, (ax, transform) in enumerate(zip(axes, transforms)):
        outline = np.array([transform(v) for v in room])
        ax.plot(*outline.T, color="0.65", label="Стены комнаты")
        for name, point in beacons.items():
            q=transform(point)
            ax.scatter(*q, marker="*", s=180, color="#1565c0", label=f"Маяк {name}" if name==target else None)
            ax.annotate(name, q, xytext=(8, 9), textcoords="offset points")
        for point, marker, label, color in [(robot,"s","Старт робота / O карты","#222222"),
                (a,"^","Начало камеры O′","#8e24aa"), (wrong,"x","Ошибка: пакет как x","#d97706")]:
            q=transform(point)
            ax.scatter(*q, marker=marker, s=65, color=color, label=label)
        start, end=transform(robot), transform(destination)
        ax.annotate("", end, start, arrowprops=dict(arrowstyle="->", lw=2, color="#222222", linestyle="-" if corrected else "--"))
        ax.plot([], [], color="#222222", linestyle="-" if corrected else "--", label="Верный маршрут" if corrected else "Ошибочный маршрут")
        for j in range(2):
            origin=transform(a); tip=transform(a+P[:,j])
            ax.annotate("", tip, origin, arrowprops=dict(arrowstyle="->", color="#8e24aa"))
            ax.annotate(f"g{j+1}", tip, xytext=(4,-13), textcoords="offset points", backgroundcolor="white")
        if k==0:
            for t in range(-5, 7):
                for j in range(2):
                    points=np.array([a + P @ (np.array([t,s]) if j==0 else np.array([s,t])) for s in [-7,7]])
                    ax.plot(*points.T, color="#8e24aa", alpha=.09)
        if decompose:
            nodes=[np.zeros(2), a, a+y[0]*P[:,0], x]
            for u,v,label in zip(nodes[:-1],nodes[1:],["a", "y₁g₁", "y₂g₂"]):
                u,v=transform(u),transform(v)
                ax.annotate("",v,u,arrowprops=dict(arrowstyle="->",color="#00838f",lw=2))
                ax.annotate(label,(u+v)/2,xytext=(5,5),textcoords="offset points",backgroundcolor="white")
        extent=np.vstack([outline, transform(wrong), transform(a), transform(robot)])
        ax.set_xlim(extent[:,0].min()-1,extent[:,0].max()+1)
        ax.set_ylim(extent[:,1].min()-1,extent[:,1].max()+1)
        ax.set_aspect("equal"); ax.grid(alpha=.2)
        ax.set_xlabel("x₁ карты" if k==0 else "y₁ камеры")
        ax.set_ylabel("x₂ карты" if k==0 else "y₂ камеры")
        ax.set_title("Карта: физическая геометрия" if k==0 else "Та же сцена: координаты камеры")
    # Один масштаб единицы для обеих панелей; в косом базисе это масштаб чисел.
    span = max(max(np.diff(ax.get_xlim())[0], np.diff(ax.get_ylim())[0]) for ax in axes)
    for ax in axes:
        cx, cy = np.mean(ax.get_xlim()), np.mean(ax.get_ylim())
        ax.set_xlim(cx-span/2, cx+span/2)
        ax.set_ylim(cy-span/2, cy+span/2)
    axes[0].legend(loc="upper left", fontsize=8)
    error=np.linalg.norm(destination-x)
    fig.suptitle(f"Маяк {target}: x={np.round(x,2)}, y={np.round(y,2)} | до цели от остановки: {error:.3f}")
    if skew:
        axes[1].text(.02,.02,"Косые координаты: длины на рисунке\nне равны физическим без метрики M",transform=axes[1].transAxes,fontsize=8,backgroundcolor="white")
    return fig

scene()
plt.show()

**После наблюдения:** промах — расстояние между остановкой и маяком, а не расстояние от камеры. Согласуется ли число на карточке с вашим прогнозом?

## 2. Как камера описывает точку · 8 минут
Стартуем в начале камеры $a$. Затем делаем $y_1$ шагов вдоль $g_1$ и $y_2$ шагов вдоль $g_2$. Столбцы $P=[g_1\ g_2]$ записаны в координатах карты. Числа $y_1,y_2$ — коэффициенты этих шагов, а не компоненты на осях карты.

**Карточка инструментов:** $x=a+Py$. Матрица перехода «от E к G» построена из нового базиса, но её действие на координаты — **G → E**. Столбцы храним в NumPy одномерными массивами `(2,)`, интерпретируя их как векторы-столбцы.

На диаграмме сопоставьте три стрелки с частями формулы. Найдите координаты маяка A у камеры самостоятельно.

In [ ]:
a = np.array([2., 0.])
P = np.array([[0., -1.], [1., 0.]])
x = np.array([1., 3.])
scene(decompose=True)
plt.show()

y_attempt = None  # ваш массив координат или выражение

<details><summary>Подсказка 1</summary>Рассмотрите стрелку от начала камеры к маяку.</details>
<details><summary>Подсказка 2</summary>В координатах карты эта стрелка равна $x-a$.</details>
<details><summary>Подсказка 3</summary>Решите $Py=x-a$: доступна функция np.linalg.solve.</details>

Запустите самопроверку после попытки.

In [ ]:
if y_attempt is None:
    print("Сначала заполните y_attempt; затем повторите эту ячейку.")
else:
    candidate = np.asarray(y_attempt)
    ok = candidate.shape == (2,) and np.allclose(a + P @ candidate, x)
    print("Верно: обратная подстановка возвращает маяк." if ok else "Проверьте направление перехода и перенос начала.")
    if ok:
        print("Ваши координаты:", candidate)

## 3. Исправьте программу навигации · 12 минут
Допишите по одной строке в двух функциях. Затем замените ошибочное присваивание `destination_attempt = packet` переводом пакета в систему карты.

Почему `P @ (x-a)` не решает задачу обратного перевода в общем случае? Что нужно знать о матрице, чтобы использовать транспонирование?

In [ ]:
def to_camera(x, a, P):
    return None  # TODO: координаты точки у камеры

def to_map(y, a, P):
    return None  # TODO: координаты точки на карте

packet = np.array([3., 1.])
destination_attempt = packet  # TODO: исправьте эту строку

<details><summary>Подсказка</summary>Проверьте смысл входа и выхода каждой функции. При возвращении на карту сначала составьте линейную комбинацию столбцов, затем добавьте начало камеры.</details>

**Самопроверка:** отдельный блок ниже проверяет также косой базис, чтобы случайное совпадение на повороте не маскировало ошибку.

In [ ]:
def verify_navigation():
    cases = [(a, P, x, np.array([3.,1.])),
             (np.array([1.,-1.]), np.array([[1.,1.],[0.,1.]]), np.array([3.,0.]), np.array([1.,1.]))]
    for origin, basis, point, coords in cases:
        yc, xc = to_camera(point, origin, basis), to_map(coords, origin, basis)
        if yc is None or xc is None:
            print("Заполните обе функции и повторите проверку.")
            return
        if not (np.asarray(yc).shape == (2,) and np.asarray(xc).shape == (2,)
                and np.allclose(yc, coords) and np.allclose(xc, point)):
            print("Переход не прошёл проверку. Учтены ли перенос и направление действия P?")
            return
    if not np.allclose(destination_attempt, x):
        print("Функции верны, но навигация всё ещё использует пакет как координаты карты.")
        return
    print("Маршрут исправлен; оба направления проверены.")
    scene(corrected=False); scene(corrected=True); plt.show()

verify_navigation()

## 4. Маршрут тот же, числа другие · 10 минут
Маяки A и B неподвижны на карте. **Предскажите**, изменятся ли их координатная разность и расстояние после поворота камеры на 30° и переноса в `(−1, 1)`.

Изменяйте параметры следующей ячейки и запускайте её заново: это основной интерактивный путь без виджетов. Сравните `corrected=False` и `True`, оба маяка, углы 90° и 30°. Для косой сетки задайте `skew=1`.

При `skew=0` единицы на обеих осях каждой панели имеют одинаковый физический масштаб. При косой сетке правая панель показывает числа в обычной координатной сетке: физические длины читаются по левой панели или через матрицу Грама.

In [ ]:
camera_origin = (-1., 1.)
camera_angle = 30.
selected_beacon = "B"  # "A" или "B"
corrected = True
skew = 0.  # 0: ортонормированный базис; 1: косой
scene(camera_origin, camera_angle, selected_beacon, corrected, skew)
plt.show()

basis = checked_basis(rotation(camera_angle) @ np.array([[1.,skew],[0.,1.]]))
x_A, x_B = np.array([1.,3.]), np.array([4.,3.])
y_A = np.linalg.solve(basis, x_A-np.array(camera_origin))
y_B = np.linalg.solve(basis, x_B-np.array(camera_origin))
print("Разность на карте:", x_B-x_A)
print("Разность у камеры:", np.round(y_B-y_A, 4))
print("Расстояние на карте:", np.linalg.norm(x_B-x_A))
print("Обычная норма разности у камеры:", np.linalg.norm(y_B-y_A))

**Ваш шаг:** раскройте скобки в $x_B-x_A=(a+Py_B)-(a+Py_A)$ и объясните, куда исчезает $a$.

**Моё объяснение:** …

<details><summary>Проверка после объяснения</summary>Одинаковый перенос сокращается: $x_B-x_A=P(y_B-y_A)$. При ортонормированных столбцах $P^TP=I$, поэтому расстояние равно 3 в обеих системах. Для исходной камеры $y_A=(3,1)^T$, $y_B=(3,-2)^T$. При косом базисе обычная норма координатной разности не обязана давать физическую длину.</details>

Смена описания неподвижных маяков — **пассивная**. Поездка робота при фиксированных осях карты — **активное** изменение положения. Для поворота новых осей новые координаты свободного вектора равны $R(\varphi)^Tx_E$; у точки дополнительно вычитается начало.

## 5. Косая сетка обманывает измеритель · ещё 15–25 минут
Пусть $g_1=(1,0)^T$, $g_2=(1,1)^T$ и координаты **перемещения** $y=(1,1)^T$. Перенос начала здесь не участвует.

**Прогноз:** совпадают ли $\|y\|$ и физическая длина $\|Py\|$? Нарисуйте сумму двух шагов до запуска.

In [ ]:
P_skew = np.array([[1.,1.],[0.,1.]])
y_shift = np.array([1.,1.])
physical_shift = P_skew @ y_shift
fig, ax = plt.subplots(figsize=(6,4), layout="constrained")
for end, style, label in [(y_shift,"--","Числа y как точка: неверная геометрия"),
                         (physical_shift,"-","Физическое перемещение Py")]:
    ax.annotate("",end,(0,0),arrowprops=dict(arrowstyle="->",linestyle=style,lw=2))
    ax.plot([],[],style,color="black",label=label)
ax.plot([0,1,2],[0,0,1],"o:",color="#1565c0",label="Шаги g₁, затем g₂")
ax.set(xlim=(-.3,2.6),ylim=(-.3,1.8),xlabel="x₁ карты",ylabel="x₂ карты",
       title=f"Норма чисел: {np.linalg.norm(y_shift):.3f}; физическая длина: {np.linalg.norm(physical_shift):.3f}")
ax.set_aspect("equal"); ax.grid(alpha=.2); ax.legend(fontsize=8)
plt.show()

**Карточка инструментов:** $M=P^TP$ — матрица Грама; $\|Py\|^2=y^TMy$. Она хранит длины базисных векторов и их скалярные произведения.

Вычислите `M_attempt` и квадрат физической длины `length2_attempt` через неё.

<details><summary>Подсказка: откуда смешанный член?</summary>$\|y_1g_1+y_2g_2\|^2=y_1^2(g_1\cdot g_1)+2y_1y_2(g_1\cdot g_2)+y_2^2(g_2\cdot g_2)$. В косом базисе средний член обычно не нулевой.</details>

In [ ]:
M_attempt = None  # TODO
length2_attempt = None  # TODO

if M_attempt is None or length2_attempt is None:
    print("Заполните матрицу Грама и квадрат длины.")
else:
    ok = (np.asarray(M_attempt).shape == (2,2)
          and np.allclose(M_attempt, [[1,1],[1,2]]) and np.isclose(length2_attempt, 5))
    print("Верно: 5 вместо наивного 2." if ok else "Проверьте смешанный член и порядок умножения.")

**Граница применимости.** Нулевое перемещение имеет длину 0 в любой системе; угол с ним и деление на его норму не определены. Если столбцы совпали, координаты не определяются однозначно. Почти параллельные столбцы могут сильно усиливать малые ошибки.

Предскажите результат для `epsilon=0`, `1e-12`, `1`. Затем запустите ячейку. Порог ниже — защитное вычислительное соглашение, а не определение линейной зависимости.

In [ ]:
epsilon = 1e-12
P_edge = np.array([[1.,1.],[0.,epsilon]])
try:
    checked_basis(P_edge)
    print("Базис прошёл защитную проверку; нулевое перемещение:", np.linalg.solve(P_edge, np.zeros(2)))
except ValueError as error:
    print(error)

### Объясните своими словами
1. Почему пакет `(3,1)` сам по себе не указывает физическую точку без сведений о системе отсчёта?
2. Почему перенос нужен для точки, но сокращается для перемещения?
3. Почему евклидова норма координат верна для повёрнутых единичных осей, но может ошибаться для косых?
4. **Перенос знания:** второй датчик использует начало $b$ и базис $Q$. Как перевести пакет первого датчика непосредственно во второй? Какие матрицы должны быть невырожденными?
5. Исследуйте совпадающие столбцы: может ли одна точка иметь несколько пакетов? Может ли точка вообще не иметь такого описания?

**Мой вывод и способ проверки:** …

**Мост к ИИ.** В реальной системе $x$ могло бы быть положением обнаруженного объекта, а $a,P$ — результатом калибровки датчика. Здесь точки и матрицы заданы вручную. Мы переводим координаты уже найденных точек: не обрабатываем изображение, не моделируем перспективную проекцию и не обучаем модель.

<details><summary>Проверка переноса — откройте после попытки</summary>Сначала $x=a+Py$, затем решите $Qz=a-b+Py$. Для координатных систем нужны невырожденные $P,Q$; непосредственно вычисление $z$ требует решения системы с $Q$.</details>

### Отдельный блок эталонов и технической самопроверки
Открывайте после собственных попыток. Этот блок не заполняет упражнения автоматически.

In [ ]:
def reference_to_camera(x, a, P):
    return np.linalg.solve(checked_basis(P), np.asarray(x)-np.asarray(a))

def reference_to_map(y, a, P):
    return np.asarray(a) + checked_basis(P) @ np.asarray(y)

def gram(P):
    P = checked_basis(P)
    return P.T @ P

def physical_distance(delta, P):
    return float(np.sqrt(max(0., np.asarray(delta) @ gram(P) @ np.asarray(delta))))

origin=np.array([2.,0.]); basis=np.array([[0.,-1.],[1.,0.]])
A=np.array([1.,3.]); B=np.array([4.,3.])
u=reference_to_camera(A, origin, basis); v=reference_to_camera(B, origin, basis)
assert np.allclose(u, [3,1]) and np.allclose(v, [3,-2])
assert np.allclose(reference_to_map(u, origin, basis), A)
assert np.allclose(basis @ u, A-origin)
assert np.allclose(basis @ (v-u), [3,0])
assert np.isclose(np.linalg.norm(u-A), np.sqrt(8))
assert np.allclose(gram(basis), np.eye(2))
assert np.isclose(physical_distance(v-u,basis), 3)
S=np.array([[1.,1.],[0.,1.]])
assert np.allclose(gram(S), [[1,1],[1,2]])
assert np.allclose(gram(S), gram(S).T) and np.all(np.linalg.eigvalsh(gram(S))>0)
assert np.isclose(physical_distance([1,1],S)**2, 5)
assert physical_distance([0,0],S)==0
for bad in [np.array([[1.,1.],[0.,0.]]), np.array([[1.,1.],[0.,1e-12]])]:
    try:
        checked_basis(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Ожидалась защита от ненадёжного базиса")
print("Все эталонные проверки пройдены.")